# 안양만안(377) AWS 파일 정리하기

**할 일은 3번 클릭이에요.** 코드는 몰라도 돼요.

1. 아래 회색 칸 왼쪽의 ▶ 버튼을 누르세요.
2. **파일 선택** 버튼이 나오면, 기상자료개방포털에서 받은 AWS CSV 파일들(2023, 2024, 2025년)을 **한꺼번에** 골라요. (Ctrl을 누른 채 클릭하면 여러 개를 고를 수 있어요.)
3. 잠시 기다리면 `aws_377.csv` 파일이 자동으로 다운로드돼요. 그 파일을 보내주면 끝!

In [ ]:
# ▶ 이 칸만 실행하세요
import pandas as pd
from google.colab import files

def read_kma_csv(path):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일을 읽을 수 없어요: {path}")

def clean_aws(raw):
    """기상청 AWS CSV → tm, aws_ta, aws_hm, aws_ws (이미 정리된 파일이면 그대로)"""
    raw.columns = [str(c).strip() for c in raw.columns]
    if {"tm", "aws_ta", "aws_hm", "aws_ws"} <= set(raw.columns):
        df = raw[["tm", "aws_ta", "aws_hm", "aws_ws"]].copy()
        df["tm"] = pd.to_datetime(df["tm"])
        return df
    def col(prefix):
        hit = [c for c in raw.columns if c.startswith(prefix)]
        if not hit:
            raise KeyError(f"'{prefix}' 열이 없어요. 이 파일의 열: {list(raw.columns)}")
        return hit[0]
    return pd.DataFrame({
        "tm":     pd.to_datetime(raw[col("일시")]),
        "aws_ta": pd.to_numeric(raw[col("기온")], errors="coerce"),
        "aws_hm": pd.to_numeric(raw[col("습도")], errors="coerce"),
        "aws_ws": pd.to_numeric(raw[col("풍속")], errors="coerce"),
    })

print("AWS CSV 파일들을 골라주세요 (여러 개 한꺼번에)")
uploaded = files.upload()

parts = []
for name in uploaded:
    df = clean_aws(read_kma_csv(name))
    print(f"  {name}: {len(df)}줄, {df.tm.min():%Y-%m-%d} ~ {df.tm.max():%Y-%m-%d}")
    parts.append(df)

aws = pd.concat(parts, ignore_index=True).drop_duplicates("tm").sort_values("tm")
aws.to_csv("aws_377.csv", index=False, encoding="utf-8-sig")
print(f"\n완료! 총 {len(aws)}줄 ({aws.tm.min():%Y-%m-%d} ~ {aws.tm.max():%Y-%m-%d})")
print("aws_377.csv 를 다운로드합니다. 이 파일을 보내주세요.")
files.download("aws_377.csv")